# Notebook 07 - Structured Outputs, Grounding, and Localization

This notebook turns visual answers into contracts: typed fields, coordinates, and audit logs that can be checked later.


## What you will learn

- how to define structured multimodal outputs with evidence fields
- how to normalize coordinates for images and pages
- how to build audit-ready output records


In [ ]:
!pip install -q numpy pandas matplotlib pydantic
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "07_structured_outputs_grounding_and_localization"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Define a grounded schema

Pydantic is a lightweight way to express the fields a model must return and the evidence each field should reference.


In [ ]:
from pydantic import BaseModel, Field

class BoundingBox(BaseModel):
    x1: float = Field(ge=0)
    y1: float = Field(ge=0)
    x2: float = Field(gt=0)
    y2: float = Field(gt=0)

class GroundedAnswer(BaseModel):
    answer: str
    evidence_id: str
    box: BoundingBox

example = GroundedAnswer(
    answer="Q4 has the highest sales",
    evidence_id="chart:bars:q4",
    box={"x1": 0.70, "y1": 0.20, "x2": 0.84, "y2": 0.84},
)
print(example.model_dump_json(indent=2))


## Step 2: Normalize raw coordinates

Normalized coordinates are easier to compare across resized images and downstream tools.


In [ ]:
def normalize_box(box, width, height):
    x1, y1, x2, y2 = box
    return {
        "x1": round(x1 / width, 4),
        "y1": round(y1 / height, 4),
        "x2": round(x2 / width, 4),
        "y2": round(y2 / height, 4),
    }

raw_box = (448, 72, 537, 302)
normalized_box = normalize_box(raw_box, width=640, height=360)
normalized_box


## Step 3: Create an audit log

Audit logs should keep the answer, the evidence location, and any validation decision in one row.


In [ ]:
audit_log = pd.DataFrame([
    {"example_id": "chart-001", "answer": "Q4 has the highest sales", "evidence_id": "chart:bars:q4", "validator": "format_check", "passed": True},
    {"example_id": "chart-001", "answer": "Q4 has the highest sales", "evidence_id": "chart:bars:q4", "validator": "grounding_check", "passed": True},
])
audit_log


## Exercises and extensions

1. Add a timestamp_span field so the same schema can handle video or audio evidence.
1. Create a second schema for document extraction with page ids and field names instead of bounding boxes.
